# Stronger LLM baseline (Qwen3-14B, Phi-4)

Same prompts, packets and parsing as the first LLM baseline, with stronger models. 76 targets × 3 conditions = 228 prompts per model. Every answer is written to `results/` in this Drive folder as soon as it is finished. If Colab disconnects, run cells 1–3 again and then the model cell that was running: finished prompts are skipped.

**Runtime:** `Runtime → Change runtime type → T4 GPU`. About 1.5–2 h per model on a T4 (an L4, if you have it, is faster).

**If you see `OutOfMemoryError`:** `Runtime → Restart session`, then run cells 1 and 3 and the model cell again. Finished prompts are kept, and a prompt that still does not fit is skipped (listed at the end) instead of stopping the run.

## 1. Mount Drive, check the GPU

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'   # must be set before torch starts: less GPU memory fragmentation
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/task_inference_llm_strong'   # change if you uploaded elsewhere
RESULTS = BASE + '/results'
import os
for f in ('config.json', 'items.csv', 'llm_lib.py', 'prompt_system.md', 'packets'):
    assert os.path.exists(os.path.join(BASE, f)), f'missing {f}'
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install

In [ ]:
%pip install -q -U transformers accelerate "bitsandbytes>=0.46.1"
import importlib.metadata as md
print({p: md.version(p) for p in ('torch', 'transformers', 'accelerate', 'bitsandbytes')})
print('If Colab offers to restart the session now, click Restart, then run cells 1 and 3 (not this cell again).')

## 3. Load the bundle

In [ ]:
import importlib.util, torch
assert torch.cuda.is_available(), 'No GPU in this session: Runtime -> Change runtime type -> T4 GPU. If no GPU is offered, the free GPU quota is used up for now (try again later, or use Kaggle).'
assert importlib.util.find_spec('bitsandbytes'), 'bitsandbytes is not installed: run cell 2, then Runtime -> Restart session, then cells 1 and 3.'
print('GPU:', torch.cuda.get_device_name(0))
import sys, importlib, json, gc, shutil
sys.path.insert(0, BASE)
import llm_lib as L
importlib.reload(L)
cfg, items = L.load_bundle(BASE)
print(cfg['version'], '| models:', cfg['models'])
print(items.groupby(['group', 'condition']).size().unstack())

def free(model_id):
    """Release GPU memory and delete the downloaded weights (~29 GB per model) so the disk does not fill up."""
    import torch
    gc.collect(); torch.cuda.empty_cache()
    shutil.rmtree(os.path.expanduser('~/.cache/huggingface/hub/models--' + model_id.replace('/', '--')), ignore_errors=True)

## 4. Smoke test: first model, 3 prompts, into `results_smoke/` (~10 min including the download)

All three lines should show `parse_ok` True.

In [ ]:
m = L.HFModel(cfg['models'][0], cfg)
L.run_model(BASE, BASE + '/results_smoke', m, limit=3)
print(L.compile_results(BASE + '/results_smoke')[['instance_id', 'condition', 'parse_ok', 'plausible_task_set', 'most_likely_task']])
del m
import torch; gc.collect(); torch.cuda.empty_cache()   # free the GPU (the download stays on disk for cell 5)

## 5. Model 1: Qwen3-14B (all 228 prompts)

Run this right after cell 4 in the same session: the model is still downloaded.

In [ ]:
m = L.HFModel(cfg['models'][0], cfg)
L.run_model(BASE, RESULTS, m)
del m; free(cfg['models'][0])

## 6. Model 2: Phi-4 (all 228 prompts)

In [ ]:
m = L.HFModel(cfg['models'][1], cfg)
L.run_model(BASE, RESULTS, m)
del m; free(cfg['models'][1])

## 7. Compile and zip for download

Download `llm_strong_results.zip` from the Drive folder and put it in the project folder.

In [ ]:
df = L.compile_results(RESULTS)
print(df.groupby(['model', 'condition']).parse_ok.mean().unstack())
shutil.make_archive(BASE + '/llm_strong_results', 'zip', RESULTS)
print('wrote', BASE + '/llm_strong_results.zip')